# WordNet Embeds Explore

This notebook collects phrase/token embeddings from HotpotQA documents, reuses a cached index when available, visualizes embeddings with PCA and t-SNE, and adds a WordNet-based definition embedding experiment.

In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import spacy
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from spacy.tokenizer import Tokenizer
from spacy.util import compile_infix_regex
from transformers import AutoModel, AutoTokenizer

from text_processing import (
    clean_text,
    encode_chunk_batch,
    extract_important_spans,
    get_token_indices_for_phrase,
    normalize_text,
)
from utils import build_hotpot_retrieval_dataset
from wordnet_explorer import WordNetExplorer

In [ ]:
file_path = "hotpot_dev_distractor_v1.json"
num_samples = 500

spacy_model_name = "en_core_web_lg"
text_encoder_name_or_path = "/home/xiaoyue/ProtoGraphRAG/deberta-v3-large"

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 8
remove_duplicate_token = True
discard_no_word = False

output_path = Path("hotpot_phrase_token_embedding_index.pkl")

In [ ]:
nlp = None
tokenizer = None
text_encoder = None


def load_custom_nlp(model_name: str):
    loaded_nlp = spacy.load(model_name)
    infixes = [pattern for pattern in loaded_nlp.Defaults.infixes if "-" not in pattern]
    infix_re = compile_infix_regex(infixes)

    loaded_nlp.tokenizer = Tokenizer(
        loaded_nlp.vocab,
        prefix_search=loaded_nlp.tokenizer.prefix_search,
        suffix_search=loaded_nlp.tokenizer.suffix_search,
        infix_finditer=infix_re.finditer,
        token_match=loaded_nlp.tokenizer.token_match,
    )
    return loaded_nlp


def load_text_encoder(name_or_path: str, device: str):
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        name_or_path,
        local_files_only=True,
        fix_mistral_regex=True,
        use_fast=True,
    )
    if not getattr(loaded_tokenizer, "is_fast", False):
        raise TypeError("encode_chunk_batch requires a fast tokenizer because it uses return_offset_mapping=True")

    loaded_text_encoder = AutoModel.from_pretrained(
        name_or_path,
        local_files_only=True,
    )
    loaded_text_encoder.to(device)
    loaded_text_encoder.eval()
    return loaded_tokenizer, loaded_text_encoder


def ensure_nlp_loaded():
    global nlp
    if nlp is None:
        nlp = load_custom_nlp(spacy_model_name)
        print(f"Loaded spaCy model: {spacy_model_name}")
    return nlp


def ensure_text_encoder_loaded():
    global tokenizer, text_encoder
    if tokenizer is None or text_encoder is None:
        tokenizer, text_encoder = load_text_encoder(text_encoder_name_or_path, device)
        print(f"Loaded text encoder on {device}")
    return tokenizer, text_encoder


if output_path.exists():
    print(f"Found cache at {output_path}. Hotpot index rebuild will be skipped.")
else:
    ensure_nlp_loaded()
    ensure_text_encoder_loaded()

In [ ]:
if output_path.exists():
    documents = None
    samples = None
    print(f"Found cache at {output_path}. Skip Hotpot dataset loading.")
else:
    file_path = "hotpot_dev_distractor_v1.json"
    documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=500)
    print(f"documents: {len(documents)}")
    print(f"samples: {len(samples)}")
    documents[0]

In [ ]:
def collect_span_embedding_records(token_embeddings, offsets, span_list, kind, document_idx, title):
    records = []
    for term, start_char, end_char in span_list:
        token_indices = get_token_indices_for_phrase(start_char, end_char, offsets)
        if not token_indices:
            continue

        embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
        records.append(
            {
                "kind": kind,
                "term": term,
                "document_idx": int(document_idx),
                "title": title,
                "span": (int(start_char), int(end_char)),
                "embedding": embedding,
            }
        )
    return records


def build_phrase_token_embedding_index(
    documents,
    nlp,
    text_encoder,
    tokenizer,
    device,
    batch_size=8,
    remove_duplicate_token=True,
    discard_no_word=False,
):
    cleaned_documents = [clean_text(doc["text"]) for doc in documents]

    store = {
        "documents": documents,
        "cleaned_documents": cleaned_documents,
        "index": defaultdict(list),
        "config": {
            "batch_size": batch_size,
            "remove_duplicate_token": remove_duplicate_token,
            "discard_no_word": discard_no_word,
            "encoder": text_encoder.__class__.__name__,
            "tokenizer": tokenizer.__class__.__name__,
            "device": device,
            "text_encoder_name_or_path": text_encoder_name_or_path,
            "spacy_model_name": spacy_model_name,
        },
    }

    phrase_occurrences = 0
    token_occurrences = 0

    for batch_start in range(0, len(documents), batch_size):
        batch_end = min(batch_start + batch_size, len(documents))
        batch_docs = documents[batch_start:batch_end]
        batch_texts = [cleaned_documents[idx] for idx in range(batch_start, batch_end)]

        batch_spans = [
            extract_important_spans(
                text,
                nlp,
                min_tokens=2,
                remove_duplicate=remove_duplicate_token,
                discard_no_word=discard_no_word,
            )
            for text in batch_texts
        ]

        token_embeddings_batch, offsets_batch = encode_chunk_batch(
            batch_texts,
            text_encoder,
            tokenizer,
            device,
        )

        for local_idx, ((phrases, tokens), token_embeddings, offsets) in enumerate(
            zip(batch_spans, token_embeddings_batch, offsets_batch)
        ):
            document_idx = batch_start + local_idx
            title = batch_docs[local_idx]["title"]

            phrase_records = collect_span_embedding_records(
                token_embeddings,
                offsets,
                phrases,
                kind="phrase",
                document_idx=document_idx,
                title=title,
            )
            token_records = collect_span_embedding_records(
                token_embeddings,
                offsets,
                tokens,
                kind="token",
                document_idx=document_idx,
                title=title,
            )

            for record in phrase_records:
                store["index"][record["term"]].append(record)
            for record in token_records:
                store["index"][record["term"]].append(record)

            phrase_occurrences += len(phrase_records)
            token_occurrences += len(token_records)

        print(f"Processed {batch_end}/{len(documents)} documents")

    store["stats"] = {
        "num_documents": len(documents),
        "num_unique_terms": len(store["index"]),
        "num_phrase_occurrences": phrase_occurrences,
        "num_token_occurrences": token_occurrences,
        "num_total_occurrences": phrase_occurrences + token_occurrences,
    }
    return store


def lookup_embeddings(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def get_document(store, document_idx):
    return store["documents"][document_idx]


def _scatter_by_kind(coords, records, title, xlabel, ylabel, annotate=False):
    color_map = {"phrase": "tab:blue", "token": "tab:orange"}
    plt.figure(figsize=(8, 6))

    for current_kind in ["phrase", "token"]:
        indices = [idx for idx, record in enumerate(records) if record["kind"] == current_kind]
        if not indices:
            continue
        plt.scatter(
            coords[indices, 0],
            coords[indices, 1],
            s=45,
            alpha=0.75,
            c=color_map[current_kind],
            label=current_kind,
        )

    if annotate:
        for idx, record in enumerate(records):
            plt.annotate(
                f"{record['kind']}:{record['document_idx']}",
                (coords[idx, 0], coords[idx, 1]),
                fontsize=8,
                alpha=0.8,
            )

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


def plot_term_embeddings(store, query_text, kind=None, perplexity=30, random_state=42, annotate=False):
    records = lookup_embeddings(
        store,
        query_text,
        kind=kind,
        include_text=False,
        include_cleaned_text=False,
    )

    if not records:
        print(f"No records found for {query_text!r}")
        return None

    if len(records) < 2:
        print(f"Only {len(records)} record found for {query_text!r}; at least 2 are required for visualization.")
        return {"records": records}

    embeddings = np.stack([record["embedding"] for record in records]).astype(np.float32)
    normalized_query = normalize_text(query_text.strip())

    pca_coords = PCA(n_components=2).fit_transform(embeddings)
    _scatter_by_kind(
        pca_coords,
        records,
        title=f"PCA projection for '{normalized_query}' ({len(records)} records)",
        xlabel="PC1",
        ylabel="PC2",
        annotate=annotate,
    )

    tsne_perplexity = min(perplexity, len(records) - 1)
    tsne_coords = TSNE(
        n_components=2,
        perplexity=tsne_perplexity,
        init="pca",
        learning_rate="auto",
        random_state=random_state,
    ).fit_transform(embeddings)
    _scatter_by_kind(
        tsne_coords,
        records,
        title=f"t-SNE projection for '{normalized_query}' ({len(records)} records, perplexity={tsne_perplexity})",
        xlabel="t-SNE-1",
        ylabel="t-SNE-2",
        annotate=annotate,
    )

    return {
        "records": records,
        "embeddings": embeddings,
        "pca_coords": pca_coords,
        "tsne_coords": tsne_coords,
    }


def collect_wordnet_definition_embeddings(
    query_text,
    explorer,
    tokenizer,
    text_encoder,
    device,
    pos=None,
    noun_only_words=True,
    exact_match_only=True,
):
    display_text = " ".join(query_text.replace("_", " ").split())
    definitions = explorer.collect_wordnet_definitions(
        query_text,
        pos=pos,
        noun_only_words=noun_only_words,
        exact_match_only=exact_match_only,
    )

    definition_sentences = [f"{display_text} is {definition}" for definition in definitions]
    if not definition_sentences:
        return {
            "query_text": query_text,
            "definitions": [],
            "definition_sentences": [],
            "records": [],
            "embedding_list": [],
        }

    token_embeddings_batch, offsets_batch = encode_chunk_batch(
        definition_sentences,
        text_encoder,
        tokenizer,
        device,
    )

    records = []
    embedding_list = []
    start_char = 0
    end_char = len(display_text)

    for definition, sentence, token_embeddings, offsets in zip(
        definitions,
        definition_sentences,
        token_embeddings_batch,
        offsets_batch,
    ):
        token_indices = get_token_indices_for_phrase(start_char, end_char, offsets)
        if not token_indices:
            continue

        embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
        embedding_list.append(embedding)
        records.append(
            {
                "query_text": query_text,
                "display_text": display_text,
                "definition": definition,
                "definition_sentence": sentence,
                "span": (start_char, end_char),
                "embedding": embedding,
            }
        )

    return {
        "query_text": query_text,
        "definitions": definitions,
        "definition_sentences": definition_sentences,
        "records": records,
        "embedding_list": embedding_list,
    }


def assign_hotpot_embeddings_to_wordnet_anchors(query_text, embedding_store, wordnet_result, kind=None):
    hotpot_records = lookup_embeddings(
        embedding_store,
        query_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=False,
    )
    anchor_records = list(wordnet_result.get("records", []))

    if not anchor_records:
        return {
            "query_text": query_text,
            "anchor_records": [],
            "hotpot_records": hotpot_records,
            "assignments": [],
            "grouped_matches": [],
            "similarity_matrix": np.empty((len(hotpot_records), 0), dtype=np.float32),
        }

    if not hotpot_records:
        grouped_matches = [
            {
                "anchor_index": anchor_idx,
                "definition": anchor["definition"],
                "definition_sentence": anchor["definition_sentence"],
                "matches": [],
            }
            for anchor_idx, anchor in enumerate(anchor_records)
        ]
        return {
            "query_text": query_text,
            "anchor_records": anchor_records,
            "hotpot_records": [],
            "assignments": [],
            "grouped_matches": grouped_matches,
            "similarity_matrix": np.empty((0, len(anchor_records)), dtype=np.float32),
        }

    hotpot_embeddings = np.stack([record["embedding"] for record in hotpot_records]).astype(np.float32)
    anchor_embeddings = np.stack([record["embedding"] for record in anchor_records]).astype(np.float32)

    hotpot_norm = hotpot_embeddings / np.clip(np.linalg.norm(hotpot_embeddings, axis=1, keepdims=True), 1e-12, None)
    anchor_norm = anchor_embeddings / np.clip(np.linalg.norm(anchor_embeddings, axis=1, keepdims=True), 1e-12, None)
    similarity_matrix = hotpot_norm @ anchor_norm.T

    assignments = []
    grouped_matches = [
        {
            "anchor_index": anchor_idx,
            "definition": anchor["definition"],
            "definition_sentence": anchor["definition_sentence"],
            "matches": [],
        }
        for anchor_idx, anchor in enumerate(anchor_records)
    ]

    for record_idx, hotpot_record in enumerate(hotpot_records):
        best_anchor_idx = int(np.argmax(similarity_matrix[record_idx]))
        similarity_scores = similarity_matrix[record_idx]
        assignment = {
            "record_index": record_idx,
            "kind": hotpot_record["kind"],
            "term": hotpot_record["term"],
            "document_idx": hotpot_record["document_idx"],
            "title": hotpot_record["title"],
            "span": hotpot_record["span"],
            "text": hotpot_record["text"],
            "best_anchor_index": best_anchor_idx,
            "best_definition": anchor_records[best_anchor_idx]["definition"],
            "best_definition_sentence": anchor_records[best_anchor_idx]["definition_sentence"],
            "best_similarity": float(similarity_scores[best_anchor_idx]),
            "all_anchor_similarities": similarity_scores.astype(np.float32).tolist(),
        }
        assignments.append(assignment)
        grouped_matches[best_anchor_idx]["matches"].append(assignment)

    for group in grouped_matches:
        group["matches"].sort(key=lambda item: item["best_similarity"], reverse=True)

    return {
        "query_text": query_text,
        "anchor_records": anchor_records,
        "hotpot_records": hotpot_records,
        "assignments": assignments,
        "grouped_matches": grouped_matches,
        "similarity_matrix": similarity_matrix,
    }


def print_wordnet_anchor_matches(match_result, max_items_per_anchor=None, max_text_chars=400):
    query_text = match_result["query_text"]
    print(f"WordNet anchor assignment for: {query_text}")
    print(f"Total anchors: {len(match_result['grouped_matches'])}")
    print(f"Total Hotpot matches: {len(match_result['assignments'])}")

    for group in match_result["grouped_matches"]:
        print("=" * 100)
        print(f"Anchor {group['anchor_index']}")
        print(f"Definition: {group['definition']}")
        print(f"Definition sentence: {group['definition_sentence']}")
        print(f"Matched embeddings: {len(group['matches'])}")

        matches_to_show = group["matches"]
        if max_items_per_anchor is not None:
            matches_to_show = matches_to_show[:max_items_per_anchor]

        if not matches_to_show:
            print("  (no matches)")
            continue

        for item_idx, match in enumerate(matches_to_show, start=1):
            preview = match["text"][:max_text_chars].replace("\n", " ")
            print(f"  [{item_idx}] similarity={match['best_similarity']:.4f} | kind={match['kind']} | title={match['title']} | document_idx={match['document_idx']} | span={match['span']}")
            print(f"      text: {preview}")

In [ ]:
if output_path.exists():
    with output_path.open("rb") as f:
        embedding_store = pickle.load(f)
    print(f"Loaded cached index from {output_path}")
else:
    ensure_nlp_loaded()
    ensure_text_encoder_loaded()
    embedding_store = build_phrase_token_embedding_index(
        documents=documents,
        nlp=nlp,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        device=device,
        batch_size=batch_size,
        remove_duplicate_token=remove_duplicate_token,
        discard_no_word=discard_no_word,
    )

    with output_path.open("wb") as f:
        pickle.dump(embedding_store, f)
    print(f"Saved new index to {output_path}")

embedding_store["stats"]

In [ ]:
query = "United States"
records = lookup_embeddings(embedding_store, query)

print(f"query={query!r}, matches={len(records)}")

if records:
    first = records[0]
    print("kind:", first["kind"])
    print("document_idx:", first["document_idx"])
    print("title:", first["title"])
    print("span in cleaned text:", first["span"])
    print("embedding shape:", first["embedding"].shape)
    print("raw text preview:")
    print(first["text"][:500])

records[:2]

In [ ]:
plot_result = plot_term_embeddings(
    embedding_store,
    query_text="United States",
    kind=None,
    perplexity=30,
    random_state=42,
    annotate=False,
)

plot_result.keys() if plot_result is not None else None

In [ ]:
ensure_text_encoder_loaded()
wordnet_explorer = WordNetExplorer()

wordnet_result = collect_wordnet_definition_embeddings(
    query_text="United States",
    explorer=wordnet_explorer,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
    device=device,
    pos=None,
    noun_only_words=True,
    exact_match_only=True,
)

wordnet_definition_embedding_list = wordnet_result["embedding_list"]
print(f"definitions found: {len(wordnet_result['definitions'])}")
print(f"embeddings collected: {len(wordnet_definition_embedding_list)}")
wordnet_result["records"][:2]

In [ ]:
anchor_match_result = assign_hotpot_embeddings_to_wordnet_anchors(
    query_text="United States",
    embedding_store=embedding_store,
    wordnet_result=wordnet_result,
    kind=None,
)

print_wordnet_anchor_matches(
    anchor_match_result,
    max_items_per_anchor=10,
    max_text_chars=500,
)

[
    {
        "anchor_index": group["anchor_index"],
        "definition": group["definition"],
        "matched_embeddings": len(group["matches"]),
    }
    for group in anchor_match_result["grouped_matches"]
]